# X4: Research Frontiers

Lessons 0-15 built one coherent, load-bearing core: neurons through
Transformers, pretraining through alignment. That core is stable; the
field built on top of it is not. This notebook is a map, not a lesson —
where the field is pushing past the dense Transformer (scaling laws,
mixture-of-experts, state-space models), where it is reaching beyond text
(multimodal models), where it is trying to understand what these models
actually compute (interpretability), and how to keep this map current
after the course ends, since every specific claim in it will start going
stale the day it is written.

## Introduction

Two things are true about frontier research at once: the core
mathematics in this course — gradients, attention, cross-entropy, the
chain rule — does not change under any of the directions below, and the
*architectural and practical choices* built on that core change on a
timescale of months. This notebook is deliberately weighted differently
from every earlier one: less derivation, because these are active
research directions without one settled textbook answer yet, and more
map-making — enough working code to make each idea concrete, and enough
context to read a new paper in each area without starting from zero.

## Setup

In [ ]:
# Fixed seeds: every stochastic step below (routing decisions, toy
# weights, synthetic data) is reproducible.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

## Scaling Laws

11a derived and empirically illustrated the headline result: test loss
falls as a smooth power law in parameters $N$, training tokens $D$, and
compute $C \approx 6ND$, with strongly diminishing returns per doubling.
What that result *predicts*, in practice, is more specific than "bigger is
better":

- **Loss is forecastable before training finishes.** Fitting the power law
  to a handful of small, cheap training runs predicts the loss of a much
  larger run *before it is trained* — this is how frontier labs decide a
  large run is worth its cost before spending it, not after.
- **Compute-optimal allocation is a real, falsifiable answer, not a rule of
  thumb.** Hoffmann et al. (2022, "Chinchilla") showed that for a fixed
  compute budget $C$, there is a specific split between $N$ and $D$ that
  minimises loss ($N \propto D \propto \sqrt{C}$) — many models trained
  before that result were substantially *undertrained* for their parameter
  count, meaning the same compute spent on a smaller model with more data
  would have scored better.
- **The curve does not predict *capabilities*, only *loss*.** A model's
  loss on next-token prediction falls smoothly and predictably; specific
  downstream skills (arithmetic, coded reasoning, following instructions)
  can appear to emerge more suddenly as loss crosses some threshold — an
  active, unsettled research question about whether "emergence" is a real
  discontinuity or an artefact of which downstream metric is used to look
  for it.

In [ ]:
# The forecasting claim above, made concrete: fit a power law to loss
# measured at small N, then check what it predicts at a much larger N
# never trained. (Reuses the exact functional form 11a derived; no new
# maths, only a new use of it -- extrapolation.)
def power_law_loss(n, n_c, alpha):
    return (n_c / n) ** alpha


# Synthetic "measured" losses at four small, cheap scales, generated from
# a true power law plus a little noise -- standing in for the four real,
# small training runs a lab would actually run before committing to a
# large one.
true_n_c, true_alpha = 3.0e6, 0.09
small_ns = np.array([1e5, 3e5, 1e6, 3e6])
rng = np.random.default_rng(SEED)
measured_losses = power_law_loss(small_ns, true_n_c, true_alpha) * (1 + rng.normal(scale=0.01, size=4))

# Fit alpha and n_c in log-log space (log L = alpha * log(n_c) - alpha * log(n)),
# i.e. ordinary least squares on (log n, log L).
slope, intercept = np.polyfit(np.log(small_ns), np.log(measured_losses), 1)
fit_alpha = -slope
fit_n_c = np.exp(intercept / fit_alpha)

large_n = 3e9  # 1000x the largest small run -- never actually trained here
predicted_loss = power_law_loss(large_n, fit_n_c, fit_alpha)
true_loss_at_large_n = power_law_loss(large_n, true_n_c, true_alpha)

print(f"fit from 4 small runs: alpha={fit_alpha:.4f} (true {true_alpha}), n_c={fit_n_c:.3e} (true {true_n_c:.3e})")
print(f"predicted loss at N={large_n:.0e} (never trained): {predicted_loss:.4f}")
print(f"true loss at that N (from the generating law):     {true_loss_at_large_n:.4f}")
print(f"relative error of the extrapolation: {abs(predicted_loss - true_loss_at_large_n) / true_loss_at_large_n:.2%}")

In [ ]:
ns_plot = np.logspace(5, 9.7, 200)
plt.figure()
plt.loglog(small_ns, measured_losses, "o", color="tab:blue", label="4 small runs (measured)")
plt.loglog(ns_plot, power_law_loss(ns_plot, fit_n_c, fit_alpha), "--", color="tab:blue", label="fit, extrapolated")
plt.loglog([large_n], [true_loss_at_large_n], "*", color="tab:red", markersize=14, label="large run (true, never trained)")
plt.xlabel("parameters $N$ (log scale)")
plt.ylabel("loss (log scale)")
plt.title("Extrapolating a scaling-law fit far beyond the fitted range")
plt.legend()
plt.tight_layout()
plt.show()

The fit fixes $\alpha$ and $N_c$ from four runs spanning barely one
order of magnitude, then extrapolates three more orders of magnitude — and
lands within a small relative error of the true value, because the
underlying process really is a power law over this range. This is the
entire empirical basis for planning a nine-figure training run around a
predicted loss number: the shape has to actually hold at the target scale,
which is itself an empirical bet, not a guarantee — the literature calls
scale ranges where the fitted law stops holding a "break" in the scaling
law, and finding one early is a live research problem in its own right.

## Beyond Dense Transformers

A **dense** Transformer, as built in 10a-11a, uses every parameter on
every token: every attention head, every feed-forward weight, computed for
every position. Two active directions relax that in different ways.

**Mixture-of-Experts (MoE)** keeps the total parameter count large but
makes the *feed-forward* sub-layer **sparse**: instead of one feed-forward
network, a block holds $E$ separate "expert" feed-forward networks and a
small learned **router** that selects only the top-$k$ experts (often
$k=1$ or $k=2$) for each token, so only a fraction $k/E$ of the
feed-forward parameters are actually used per token. The payoff is
capacity decoupled from per-token compute: an MoE model can hold far more
total parameters than a dense model at the *same* inference FLOPs per
token, because most of those parameters sit idle for any given token.

In [ ]:
# A minimal top-1 MoE feed-forward layer: real routing, real sparsity,
# real FLOP accounting -- at a toy size, for concrete numbers rather than
# an abstract claim.
class Top1MoE(nn.Module):
    def __init__(self, d_model, d_ff, num_experts):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(num_experts)
        )

    def forward(self, x):
        # x: (tokens, d_model). Each token is routed to exactly one expert.
        gate_logits = self.router(x)
        chosen = gate_logits.argmax(dim=-1)  # (tokens,)
        out = torch.zeros_like(x)
        for e, expert in enumerate(self.experts):
            mask = chosen == e
            if mask.any():
                out[mask] = expert(x[mask])
        return out, chosen


D_MODEL, D_FF, NUM_EXPERTS, N_TOKENS = 32, 64, 8, 256
moe = Top1MoE(D_MODEL, D_FF, NUM_EXPERTS)
tokens = torch.randn(N_TOKENS, D_MODEL)
moe_out, routing = moe(tokens)

dense_ffn_params = 2 * D_MODEL * D_FF  # one expert-sized dense FFN, for comparison
moe_total_params = sum(p.numel() for p in moe.experts.parameters())
active_experts_per_token = 1
flops_per_token_dense_equiv = 2 * dense_ffn_params  # a single dense FFN this size, every token
flops_per_token_moe = 2 * (dense_ffn_params) * active_experts_per_token  # only 1 of NUM_EXPERTS runs

print(f"MoE holds {moe_total_params:,} feed-forward parameters across {NUM_EXPERTS} experts")
print(f"a same-size DENSE feed-forward layer would hold only {dense_ffn_params:,} parameters")
print(f"but per-token compute is identical: {flops_per_token_moe:,} FLOPs (MoE, 1 expert) "
      f"vs. {flops_per_token_dense_equiv:,} FLOPs (dense)")
print(f"routing is genuinely non-uniform for random init: expert usage counts = {torch.bincount(routing, minlength=NUM_EXPERTS).tolist()}")

$8\times$ the feed-forward capacity of a dense layer of the same
per-expert size, at *exactly* the same per-token compute — this is the MoE
trade in one number. The routing counts above are already non-uniform
from a randomly initialised router alone, before any training pushes
tokens toward experts that specialise for them; a well-known practical
failure mode is **router collapse**, where training pushes almost every
token to the same one or two experts, which is why production MoE systems
(Switch Transformer, Mixtral) add an explicit load-balancing loss term
that penalises uneven expert usage during training.

**State-space models (SSMs)**, the architecture behind S4 and Mamba,
relax the *sequence-mixing* mechanism instead of the feed-forward layer.
Self-attention computes, for every one of $T$ positions, a comparison
against all $T$ positions — $O(T^2)$ compute and memory, the cost 9a/10a
built entirely on top of. An SSM instead carries a **fixed-size hidden
state** $h_t$ forward through the sequence with a linear recurrence,
exactly like 7a's RNN cell but restricted to a linear update (which is
what makes a closed-form, highly parallel training algorithm possible —
the actual technical contribution of S4/Mamba over a plain RNN):

$$h_t = A h_{t-1} + B u_t, \qquad y_t = C h_t,$$

for learned matrices $A, B, C$ and input $u_t$. Because $h_t$ has *fixed*
size independent of $T$, both compute and memory per step are $O(1)$,
making the whole sequence $O(T)$ instead of attention's $O(T^2)$ — the
entire motivation for revisiting a recurrence-shaped architecture after
attention displaced RNNs in 9a/9b.

In [ ]:
# The O(T) vs. O(T^2) claim, measured directly rather than asserted:
# count the scalar multiply-adds each mechanism performs to mix
# information across a sequence of length T, for growing T.
def attention_mixing_flops(T, d_model):
    return 2 * T * T * d_model  # Q @ K^T and weights @ V, both O(T^2 * d_model)


def ssm_mixing_flops(T, d_state):
    return T * (2 * d_state * d_state)  # one fixed-size A@h_{t-1} update per step, O(T)


d_model, d_state = 64, 64
Ts = np.array([32, 64, 128, 256, 512, 1024, 2048, 4096])
attn_flops = attention_mixing_flops(Ts, d_model)
ssm_flops = ssm_mixing_flops(Ts, d_state)

plt.figure()
plt.loglog(Ts, attn_flops, "o-", label="self-attention (O(T^2))")
plt.loglog(Ts, ssm_flops, "o-", label="SSM recurrence (O(T))")
plt.xlabel("sequence length $T$ (log scale)")
plt.ylabel("sequence-mixing FLOPs (log scale)")
plt.title("Quadratic vs. linear sequence-mixing cost")
plt.legend()
plt.tight_layout()
plt.show()

crossover_ratio = attn_flops[-1] / ssm_flops[-1]
print(f"at T={Ts[-1]}, attention costs {crossover_ratio:.0f}x the SSM's sequence-mixing FLOPs")

The two curves are parallel on log-log axes only because both are
plotted against the same $T$ range on a log scale — attention's slope is
visibly steeper, and the gap between them widens without bound as $T$
grows, which is exactly the practical motivation for SSMs at very long
context lengths (whole books, genomes, audio) where $O(T^2)$ attention
becomes the dominant cost. The trade is not free: a fixed-size $h_t$ is a
lossy summary of everything before it, so an SSM's ability to retrieve a
specific fact from arbitrarily far back in a long context is an active
area of empirical study and architectural refinement (selective
state-space updates, hybrid attention+SSM layers), not a settled
solution.

## Multimodal Models

Every model in this course has consumed one modality — text tokens or
image patches — at a time. A **multimodal** model instead aligns two or
more modalities into a **shared representation space**, so that, for
instance, an image and a caption describing it end up with similar
embeddings. **CLIP** (Radford et al., 2021) is the architecture most
subsequent multimodal work builds on: an image encoder and a text encoder,
trained *jointly* with a **contrastive objective** — for a batch of
(image, caption) pairs, push each image's embedding close to its own
caption's embedding and far from every other caption's embedding in the
same batch, and symmetrically for text-to-image. Vision-language models
that can *converse* about an image (GPT-4V-style, LLaVA) typically extend
this further: a pretrained vision encoder's patch embeddings (8a/8b's
tokenisation, applied to image patches instead of text) are projected into
the *same* embedding space a pretrained language model already uses, so
the language model attends over image-patch tokens exactly as it attends
over text tokens — no architectural change to the Transformer itself, only
to what gets embedded and handed to it.

In [ ]:
# A tiny, from-scratch illustration of CLIP's actual training signal --
# the contrastive loss -- computed exactly as it would be for a real batch,
# just with toy random embeddings standing in for real image/text encoders.
def clip_contrastive_loss(image_embeds, text_embeds, temperature=0.07):
    image_embeds = F.normalize(image_embeds, dim=-1)
    text_embeds = F.normalize(text_embeds, dim=-1)
    logits = image_embeds @ text_embeds.T / temperature  # (batch, batch)
    labels = torch.arange(logits.shape[0])  # pair i's positive match is index i
    loss_i2t = F.cross_entropy(logits, labels)       # each image should match its own caption
    loss_t2i = F.cross_entropy(logits.T, labels)     # each caption should match its own image
    return (loss_i2t + loss_t2i) / 2, logits


torch.manual_seed(SEED)
BATCH, EMBED_DIM = 6, 16
# "Aligned" embeddings: pair i's image and text vectors are made deliberately
# similar, standing in for what a trained encoder pair would produce.
base = torch.randn(BATCH, EMBED_DIM)
aligned_image_embeds = base + 0.1 * torch.randn(BATCH, EMBED_DIM)
aligned_text_embeds = base + 0.1 * torch.randn(BATCH, EMBED_DIM)
random_text_embeds = torch.randn(BATCH, EMBED_DIM)  # an untrained/misaligned pairing, for contrast

aligned_loss, aligned_logits = clip_contrastive_loss(aligned_image_embeds, aligned_text_embeds)
random_loss, _ = clip_contrastive_loss(aligned_image_embeds, random_text_embeds)

print(f"contrastive loss, image/text embeddings genuinely aligned:   {aligned_loss.item():.4f}")
print(f"contrastive loss, text embeddings replaced with random noise: {random_loss.item():.4f}")
print(f"diagonal (correct-pair) similarity, mean: {aligned_logits.diag().mean().item():.2f}")
print(f"off-diagonal (wrong-pair) similarity, mean: {(aligned_logits.sum() - aligned_logits.diag().sum()).item() / (BATCH * BATCH - BATCH):.2f}")
assert aligned_loss.item() < random_loss.item()

Aligned pairs give a visibly lower contrastive loss than random
pairs, and the diagonal (correct pairings) scores higher similarity than
the off-diagonal (incorrect pairings) — exactly the signal that, at real
scale (CLIP trained on 400 million image-caption pairs scraped from the
web), pulls an image encoder and a text encoder into a genuinely shared
space with no manually labelled correspondence beyond "this caption came
with this image," the same self-supervision-from-naturally-occurring-pairs
idea 11a's pretraining objective uses for text alone.

## Interpretability

A model that predicts well is not the same as a model whose
computation is understood. **Interpretability** research asks what a
trained network's weights and activations actually represent and compute
— increasingly urgent as these models are deployed for decisions people
rely on. Three broad approaches, roughly from least to most invasive:

- **Probing**: train a small, separate classifier on a frozen model's
  internal activations to test whether some human-interpretable property
  (part of speech, sentiment, factual truth) is *linearly decodable* from
  them — evidence the model represents that property somewhere, without
  claiming to explain how it is used.
- **Attention visualisation**: inspect which positions a trained head
  attends to, as a cheap, visual first pass at "what is this component
  looking at." It is popular precisely because 9a/10a already computes
  attention weights as part of the forward pass — no extra machinery
  needed. It is also contested: attention weight is not proven to equal
  causal importance to the output (a head can attend heavily to a token
  whose value vector contributes almost nothing), so it is treated here as
  one weak, easy-to-compute probe among several, not a full explanation.
- **Mechanistic interpretability**: reverse-engineer a specific, reusable
  computational circuit inside a trained network — the current frontier,
  exemplified by work identifying **induction heads** (attention head
  pairs that implement "if this token followed that token before, predict
  it follows again," a genuine in-context-learning mechanism found inside
  trained Transformers) and, more recently, **sparse autoencoders** trained
  on a model's activations to decompose them into a larger set of sparser,
  more individually interpretable "features" than the raw activation
  dimensions provide.

In [ ]:
# Attention visualisation as a cheap, easy-to-compute (and contested)
# interpretability probe: reuse 9a/10a's exact scaled dot-product
# attention, on a short sequence engineered so the "right" answer is
# knowable in advance -- each token should attend most to the position it
# is a designed match for.
def softmax(x, axis=-1):
    shifted = x - x.max(axis=axis, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=axis, keepdims=True)


tokens_demo = ["The", "cat", "sat", "on", "the", "mat"]
T = len(tokens_demo)
d = T
rng2 = np.random.default_rng(SEED)
# Hand-built key/query vectors: every token's key is an orthogonal basis
# vector (so no two keys can be confused for each other by chance), and
# "sat" (index 2) is engineered to query exactly like "cat" (index 1), and
# "mat" (index 5) exactly like "the" (index 4) -- a designed ground truth,
# not a hoped-for one, to check the visualisation against.
K = np.eye(T, d) + rng2.normal(scale=0.01, size=(T, d))
Q = K.copy()
Q[2] = K[1] + rng2.normal(scale=0.01, size=d)   # "sat" queries like "cat"
Q[5] = K[4] + rng2.normal(scale=0.01, size=d)   # "mat" queries like "the"

scores = (Q @ K.T) / np.sqrt(d)
weights = softmax(scores)

plt.figure(figsize=(5, 4.5))
plt.imshow(weights, cmap="viridis")
plt.xticks(range(T), tokens_demo)
plt.yticks(range(T), tokens_demo)
plt.xlabel("attending TO (key)")
plt.ylabel("attending FROM (query)")
plt.title("Attention weights: a cheap interpretability probe")
plt.colorbar(label="attention weight")
plt.tight_layout()
plt.show()

print(f"'sat' attends most strongly to: {tokens_demo[weights[2].argmax()]} (designed match: 'cat')")
print(f"'mat' attends most strongly to: {tokens_demo[weights[5].argmax()]} (designed match: 'the')")
assert weights[2].argmax() == 1 and weights[5].argmax() == 4

The visualisation correctly recovers both designed matches here —
because the example was engineered so that attention weight *is* the
whole story. A real trained model offers no such guarantee: the caveat in
the bullet above (attention weight is not proven to equal causal
importance) is precisely why mechanistic interpretability goes further,
verifying a hypothesised circuit by *intervening* on it (ablating or
patching an activation and checking the output changes as the hypothesis
predicts) rather than reading attention weights alone and stopping there.

## Staying Current

Every specific number and technique above will age; the skill that
does not is knowing where to keep looking. In roughly the order a
practitioner actually uses them:

- **arXiv (cs.LG, cs.CL, cs.CV)** is where nearly all of this research is
  posted first, often a year or more before formal peer review — the
  primary source, not a summary of one.
- **Conference proceedings** — NeurIPS, ICML, and ICLR for the general
  field; ACL and EMNLP for language-specific work; CVPR for vision — are
  where arXiv preprints are formally reviewed and where the annual
  "state of the field" snapshots are easiest to find.
- **Paper-to-code aggregators** (Papers With Code and similar sites)
  connect a paper to a runnable implementation and a leaderboard, which is
  often the fastest way to judge whether a new method's claims replicate.
- **Reference texts** for the material this course assumed or built on
  top of: *Deep Learning* (Goodfellow, Bengio, Courville, 2016) for
  foundations, and Anthropic's, OpenAI's, and Google DeepMind's own
  published research blogs for frontier-lab-specific technique writeups
  (RLHF variants, scaling methodology, interpretability) that often appear
  there before or alongside a paper.
- **Reproducing a small piece of a new paper**, the way every notebook in
  this course reproduces a result from scratch, remains the most reliable
  way to tell whether a claimed result is real, exactly as it has been for
  every lesson so far — the skill this course taught does not go out of
  date even when every specific architecture in this notebook does.

## Key Takeaways

- **Scaling laws predict loss before a run finishes**, not just
  describe it after — fitting a power law to a handful of cheap small runs
  extrapolated within a small error to a size never trained, measured
  directly above; they say nothing about *when* a specific downstream
  capability appears.
- **Mixture-of-experts and state-space models relax different parts of the
  dense Transformer**: MoE trades a sparse, routed feed-forward layer for
  far more total capacity at the same per-token compute (measured: 8x
  capacity, identical FLOPs); SSMs trade $O(T^2)$ attention for an $O(T)$
  linear recurrence, at the cost of summarising history into a fixed-size
  state instead of attending to it directly.
- **Multimodal models align modalities with the same contrastive,
  self-supervised idea 11a's pretraining objective uses for text alone** —
  push a true (image, caption) pair's embeddings together and every
  mismatched pair apart, verified directly on toy aligned vs. random
  embeddings above.
- **Interpretability ranges from cheap and contested (attention
  visualisation) to invasive and rigorous (mechanistic interpretability
  with causal interventions)** — attention weight alone is a probe, not
  proof of causal importance, which is exactly why the field is moving
  toward intervention-based verification.
- **The one skill in this notebook that does not go stale is reproducing a
  claimed result from scratch** — arXiv, the major conferences, and
  paper-to-code aggregators are where to find the next claim worth
  reproducing.